In [ ]:
import sys
import subprocess

def ensure_package(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

ensure_package('datasets')
ensure_package('pandas')
ensure_package('numpy')
ensure_package('scikit-learn', 'sklearn')

In [ ]:
import random
import time
import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

In [ ]:
dataset = load_dataset('emotion')
print(dataset)
print('Splits:', list(dataset.keys()))
for split in dataset.keys():
    print(split, len(dataset[split]))

In [ ]:
label_feature = dataset['train'].features['label']
label_names = label_feature.names
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in id2label.items()}
print('Labels:', id2label)

train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()

def enrich_dataframe(df):
    out = df.copy()
    out['label_name'] = out['label'].map(id2label)
    out['text'] = out['text'].astype(str)
    out['text_clean'] = out['text'].str.strip().str.lower()
    out['char_len'] = out['text'].str.len()
    out['word_len'] = out['text'].str.split().str.len()
    out['unique_word_len'] = out['text_clean'].str.split().apply(lambda x: len(set(x)) if isinstance(x, list) else 0)
    out['avg_word_len'] = out['text_clean'].str.split().apply(lambda x: float(np.mean([len(w) for w in x])) if isinstance(x, list) and len(x) > 0 else 0.0)
    out['exclamation_count'] = out['text'].str.count('!')
    out['question_count'] = out['text'].str.count(r'\?')
    out['uppercase_ratio'] = out['text'].apply(lambda s: (sum(1 for c in s if c.isupper()) / max(1, sum(1 for c in s if c.isalpha()))))
    return out

train_df = enrich_dataframe(train_df)
val_df = enrich_dataframe(val_df)
test_df = enrich_dataframe(test_df)

print(train_df.head(10).to_string(index=False))

In [ ]:
print('Train shape:', train_df.shape)
print('Validation shape:', val_df.shape)
print('Test shape:', test_df.shape)

print('\nMissing values by split:')
for name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    print(f'--- {name} ---')
    print(df.isna().sum())

print('\nDuplicate text counts by split:')
for name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    dup_count = df['text_clean'].duplicated().sum()
    unique_count = df['text_clean'].nunique()
    print(f'{name}: duplicates={dup_count}, unique_texts={unique_count}, total={len(df)}')

print('\nSample rows from each split:')
for name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    print(f'--- {name} sample ---')
    print(df[['text', 'label_name', 'char_len', 'word_len', 'unique_word_len', 'avg_word_len']].head(5).to_string(index=False))

In [ ]:
def class_balance_table(df, split_name):
    counts = df['label_name'].value_counts().sort_index()
    pct = (counts / len(df) * 100).round(2)
    out = pd.DataFrame({'count': counts, 'percent': pct})
    out['split'] = split_name
    return out.reset_index().rename(columns={'index': 'label_name'})

train_balance = class_balance_table(train_df, 'train')
val_balance = class_balance_table(val_df, 'validation')
test_balance = class_balance_table(test_df, 'test')

print('Train label distribution:')
print(train_balance.to_string(index=False))
print('\nValidation label distribution:')
print(val_balance.to_string(index=False))
print('\nTest label distribution:')
print(test_balance.to_string(index=False))

combined_balance = pd.concat([train_balance, val_balance, test_balance], ignore_index=True)
pivot_balance = combined_balance.pivot(index='label_name', columns='split', values='count').fillna(0).astype(int)
pivot_balance_pct = combined_balance.pivot(index='label_name', columns='split', values='percent').fillna(0)

print('\nClass count comparison across splits:')
print(pivot_balance)
print('\nClass percentage comparison across splits:')
print(pivot_balance_pct)

imbalance_ratio = train_balance['count'].max() / train_balance['count'].min()
print(f'\nTrain split imbalance ratio (max class / min class): {imbalance_ratio:.3f}')

In [ ]:
length_cols = ['char_len', 'word_len', 'unique_word_len', 'avg_word_len', 'exclamation_count', 'question_count', 'uppercase_ratio']

print('Overall text feature statistics (train):')
print(train_df[length_cols].describe().T)

print('\nText feature statistics by label (train):')
group_stats = train_df.groupby('label_name')[length_cols].agg(['mean', 'median', 'min', 'max']).round(3)
print(group_stats)

print('\nShortest training examples:')
print(train_df.sort_values(['word_len', 'char_len']).head(15)[['text', 'label_name', 'char_len', 'word_len']].to_string(index=False))

print('\nLongest training examples:')
print(train_df.sort_values(['word_len', 'char_len'], ascending=False).head(15)[['text', 'label_name', 'char_len', 'word_len']].to_string(index=False))

In [ ]:
def top_tokens_by_label(df, top_n=15):
    rows = []
    for label in sorted(df['label_name'].unique()):
        subset = df[df['label_name'] == label]
        token_series = subset['text_clean'].str.split().explode()
        token_counts = token_series.value_counts().head(top_n)
        for token, count in token_counts.items():
            rows.append({'label_name': label, 'token': token, 'count': int(count)})
    return pd.DataFrame(rows)

token_summary = top_tokens_by_label(train_df, top_n=12)
print('Top tokens by label from train split:')
for label in label_names:
    print(f'--- {label} ---')
    print(token_summary[token_summary['label_name'] == label].to_string(index=False))

In [ ]:
X_train = train_df['text_clean'].tolist()
y_train = train_df['label'].tolist()
X_val = val_df['text_clean'].tolist()
y_val = val_df['label'].tolist()
X_test = test_df['text_clean'].tolist()
y_test = test_df['label'].tolist()

model = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        random_state=SEED,
        n_jobs=None
    ))
])

start_train = time.time()
model.fit(X_train, y_train)
train_time = time.time() - start_train

start_val = time.time()
val_preds = model.predict(X_val)
val_infer_time = time.time() - start_val

start_test = time.time()
test_preds = model.predict(X_test)
test_infer_time = time.time() - start_test

val_acc = accuracy_score(y_val, val_preds)
val_f1 = f1_score(y_val, val_preds, average='macro')
test_acc = accuracy_score(y_test, test_preds)
test_f1 = f1_score(y_test, test_preds, average='macro')

print(f'Training time: {train_time:.3f}s')
print(f'Validation inference time: {val_infer_time:.3f}s')
print(f'Test inference time: {test_infer_time:.3f}s')
print(f'Validation Accuracy: {val_acc:.4f}')
print(f'Validation Macro F1: {val_f1:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test Macro F1: {test_f1:.4f}')

In [ ]:
print('Validation classification report:')
print(classification_report(y_val, val_preds, target_names=label_names, digits=4))

print('Test classification report:')
print(classification_report(y_test, test_preds, target_names=label_names, digits=4))

cm = confusion_matrix(y_test, test_preds)
cm_df = pd.DataFrame(cm, index=[f'true_{x}' for x in label_names], columns=[f'pred_{x}' for x in label_names])
print('Test confusion matrix:')
print(cm_df)

In [ ]:
val_probas = model.predict_proba(X_val)
test_probas = model.predict_proba(X_test)
val_conf = val_probas.max(axis=1)
test_conf = test_probas.max(axis=1)

diagnostics_df = pd.DataFrame({
    'split': ['validation', 'test'],
    'accuracy': [val_acc, test_acc],
    'macro_f1': [val_f1, test_f1],
    'avg_confidence': [float(val_conf.mean()), float(test_conf.mean())],
    'median_confidence': [float(np.median(val_conf)), float(np.median(test_conf))],
    'min_confidence': [float(val_conf.min()), float(test_conf.min())],
    'max_confidence': [float(val_conf.max()), float(test_conf.max())]
})
print(diagnostics_df.to_string(index=False))

test_pred_df = test_df[['text', 'text_clean', 'label', 'label_name', 'char_len', 'word_len']].copy()
test_pred_df['pred'] = test_preds
test_pred_df['pred_name'] = test_pred_df['pred'].map(id2label)
test_pred_df['confidence'] = test_conf
test_pred_df['correct'] = test_pred_df['label'] == test_pred_df['pred']

print('\nLowest-confidence correct test predictions:')
print(test_pred_df[test_pred_df['correct']].sort_values('confidence').head(20)[['text', 'label_name', 'pred_name', 'confidence', 'word_len']].to_string(index=False))

print('\nHighest-confidence incorrect test predictions:')
print(test_pred_df[~test_pred_df['correct']].sort_values('confidence', ascending=False).head(20)[['text', 'label_name', 'pred_name', 'confidence', 'word_len']].to_string(index=False))

In [ ]:
tfidf = model.named_steps['tfidf']
clf = model.named_steps['clf']
feature_names = np.array(tfidf.get_feature_names_out())
coef = clf.coef_

top_feature_rows = []
for class_idx, class_name in enumerate(label_names):
    class_coef = coef[class_idx]
    top_pos_idx = np.argsort(class_coef)[-15:][::-1]
    top_neg_idx = np.argsort(class_coef)[:15]
    for rank, idx in enumerate(top_pos_idx, start=1):
        top_feature_rows.append({
            'label_name': class_name,
            'direction': 'positive',
            'rank': rank,
            'feature': feature_names[idx],
            'weight': float(class_coef[idx])
        })
    for rank, idx in enumerate(top_neg_idx, start=1):
        top_feature_rows.append({
            'label_name': class_name,
            'direction': 'negative',
            'rank': rank,
            'feature': feature_names[idx],
            'weight': float(class_coef[idx])
        })

top_features_df = pd.DataFrame(top_feature_rows)
print('Top positive and negative TF-IDF features per class:')
for class_name in label_names:
    print(f'--- {class_name} positive features ---')
    print(top_features_df[(top_features_df['label_name'] == class_name) & (top_features_df['direction'] == 'positive')][['rank', 'feature', 'weight']].to_string(index=False))
    print(f'--- {class_name} negative features ---')
    print(top_features_df[(top_features_df['label_name'] == class_name) & (top_features_df['direction'] == 'negative')][['rank', 'feature', 'weight']].to_string(index=False))

In [ ]:
results_df = pd.DataFrame([
    {
        'model': 'TF-IDF + LogisticRegression',
        'dataset': 'emotion',
        'val_accuracy': val_acc,
        'val_macro_f1': val_f1,
        'test_accuracy': test_acc,
        'test_macro_f1': test_f1,
        'train_time_sec': train_time,
        'val_inference_sec': val_infer_time,
        'test_inference_sec': test_infer_time,
        'n_train': len(train_df),
        'n_validation': len(val_df),
        'n_test': len(test_df),
        'n_classes': len(label_names)
    }
])
print(results_df.to_string(index=False))

In [ ]:
error_df = test_df[['text', 'text_clean', 'label', 'label_name', 'char_len', 'word_len', 'unique_word_len', 'avg_word_len']].copy()
error_df['pred'] = test_preds
error_df['pred_name'] = error_df['pred'].map(id2label)
error_df['confidence'] = test_conf
error_df['correct'] = error_df['label'] == error_df['pred']

misclassified = error_df[~error_df['correct']].copy()
print('Total test examples:', len(error_df))
print('Correct test examples:', int(error_df['correct'].sum()))
print('Misclassified test examples:', len(misclassified))

print('\nTop confusion pairs:')
print(misclassified.groupby(['label_name', 'pred_name']).size().sort_values(ascending=False).head(20))

print('\nMisclassification stats by true label:')
misclass_by_true = misclassified.groupby('label_name').size().rename('misclassified_count').reset_index()
true_counts = error_df.groupby('label_name').size().rename('total_count').reset_index()
misclass_summary = true_counts.merge(misclass_by_true, on='label_name', how='left').fillna(0)
misclass_summary['misclassified_count'] = misclass_summary['misclassified_count'].astype(int)
misclass_summary['misclassification_rate'] = (misclass_summary['misclassified_count'] / misclass_summary['total_count']).round(4)
print(misclass_summary.to_string(index=False))

print('\nSample misclassifications:')
print(misclassified[['text', 'label_name', 'pred_name', 'confidence', 'word_len']].head(25).to_string(index=False))

In [ ]:
def predict_emotion(texts):
    cleaned = [str(t).strip().lower() for t in texts]
    pred_ids = model.predict(cleaned)
    probas = model.predict_proba(cleaned)
    confs = probas.max(axis=1)
    return pd.DataFrame({
        'text': texts,
        'pred_label_id': pred_ids,
        'pred_label': [id2label[i] for i in pred_ids],
        'confidence': confs
    })

sample_texts = [
    'i feel amazing and grateful today',
    'i am really upset and angry about what happened',
    'i miss my friends and feel lonely',
    'i am scared about tomorrow',
    'this was such a lovely surprise'
]

print(predict_emotion(sample_texts).to_string(index=False))